In [18]:
#%% [markdown]
# === CONFIGURACIÓN BÁSICA ===
# Cambia SOLO estas variables y corre todo el notebook.
# Si estás en este entorno y subiste tus patrones, puedes usar:
# PATTERNS_FILE = "/mnt/data/express.py"

INPUT_DIR         = r"D:\historias\_01_historias\LETRA A\ACTA N° 70\1"
PATTERNS_FILE     = r"C:\Users\juans\Downloads\define\express.py" 
OUTPUT_XLSX       = r"C:\Users\juans\Documents\version_final_historias laborales\answer_ocr\inventario_ocr.xlsx"

# OCR
ENGINE            = "auto"         # "auto" | "tesseract" | "paddle"
LANG              = "spa+eng"      # ej. "spa", "spa+eng"
TESSERACT_CMD     = None           # ej. r"C:\Program Files\Tesseract-OCR\tesseract.exe" (o None)
DPI               = 250            # 250-300 recomendado
MIN_LEN_FOR_TEXT  = 30             # si texto embebido < este tamaño => hacer OCR

# CLASIFICACIÓN por regex
MIN_SCORE         = 1.2            # subir si hay falsos positivos
MIN_UNIQUE        = 1              # # de patrones "únicos" mínimos por etiqueta
AMBIG_MARGIN      = 0.15           # margen para marcar como AMB si top1 - top2 es bajo

# Recursividad
RECURSIVE         = True           # buscar PDFs en subcarpetas


In [19]:
#%%
from __future__ import annotations
import os, re, sys, json, math, unicodedata, logging
from pathlib import Path
from typing import Dict, List, Iterable, Tuple, Optional, Any

import pandas as pd

try:
    import fitz  # PyMuPDF
except Exception as e:
    raise SystemExit("Necesitas PyMuPDF: pip install pymupdf") from e

from PIL import Image

try:
    import pytesseract
except Exception:
    pytesseract = None

try:
    import yaml  # opcional (para .yml/.yaml)
except Exception:
    yaml = None

try:
    from tqdm import tqdm
except Exception:
    tqdm = None

# Preprocesado OCR (opcional)
try:
    import cv2, numpy as np
except Exception:
    cv2, np = None, None

# PaddleOCR (opcional GPU)
try:
    from paddleocr import PaddleOCR
except Exception:
    PaddleOCR = None

# Logging
LOG = logging.getLogger("ocr_regex_nb")
if not LOG.handlers:
    _h = logging.StreamHandler(stream=sys.stdout)
    _f = logging.Formatter("[%(levelname)s] %(message)s")
    _h.setFormatter(_f)
    LOG.addHandler(_h)
LOG.setLevel(logging.INFO)

def normalize_patterns_keys(pats: PatternsDict) -> PatternsDict:
    for lbl, d in pats.items():
        # si viene como lista simple, conviértelo a dict
        if isinstance(d, list):
            d = {"any": d}
            pats[lbl] = d
        # asegura las tres llaves
        d.setdefault("any", [])
        d.setdefault("must", [])
        d.setdefault("exclude", [])
    if "SIN_CLASIFICAR" not in pats:
        pats["SIN_CLASIFICAR"] = {"any": [], "must": [], "exclude": []}
    return pats

#%%
RegexList = List[re.Pattern]
PatternsDict = Dict[str, Dict[str, RegexList]]  # {label: {any:[], must:[], exclude:[]}}

_COMPILE_FLAGS = re.IGNORECASE | re.UNICODE
_DEF_KEYS = ("any", "must", "exclude")

# Parser tolerante para .py con re.compile "roto" (re,compile / re.compil...)
_COMPILE_CALL = re.compile(r"re\W*c\w*?le\s*\(\s*(r?[\"'])(.+?)\1", re.IGNORECASE | re.UNICODE)
_LABEL_START  = re.compile(r"^\s*([\"'])([^\"']+)\1\s*:\s*\[", re.UNICODE)
_LIST_END     = re.compile(r"^\s*\],?\s*$")

def _ensure_patterns_dict(raw: Dict[str, Any]) -> PatternsDict:
    out: PatternsDict = {}
    for label, value in raw.items():
        d = {k: [] for k in _DEF_KEYS}
        if isinstance(value, dict):
            # {'any': [...], 'must': [...], 'exclude': [...]}
            for k in _DEF_KEYS:
                lst = value.get(k) or []
                d[k] = [re.compile(p, _COMPILE_FLAGS) if isinstance(p, str) else p for p in lst]
        elif isinstance(value, list):
            d["any"] = [re.compile(p, _COMPILE_FLAGS) if isinstance(p, str) else p for p in value]
        else:
            continue
        out[label] = d
    if "SIN_CLASIFICAR" not in out:
        out["SIN_CLASIFICAR"] = {k: [] for k in _DEF_KEYS}
    return out

def load_patterns(patterns_path: str | Path) -> PatternsDict:
    p = Path(patterns_path)
    if not p.exists():
        raise FileNotFoundError(f"No existe el archivo de patrones: {p}")
    ext = p.suffix.lower()

    if ext == ".py":
        # import limpio
        try:
            import importlib.util
            spec = importlib.util.spec_from_file_location("user_patterns", str(p))
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            if hasattr(mod, "OCR_PATTERN") and isinstance(mod.OCR_PATTERN, dict):
                LOG.info("Patrones PY: usando OCR_PATTERN del módulo.")
                return _ensure_patterns_dict(mod.OCR_PATTERN)
            if hasattr(mod, "OCR_PATTERN_RAW") and isinstance(mod.OCR_PATTERN_RAW, dict):
                LOG.info("Patrones PY: usando OCR_PATTERN_RAW.")
                compiled = {lbl: {k: [] for k in _DEF_KEYS} for lbl in mod.OCR_PATTERN_RAW}
                for lbl, lst in mod.OCR_PATTERN_RAW.items():
                    compiled[lbl]["any"] = [re.compile(s, _COMPILE_FLAGS) for s in lst]
                if "SIN_CLASIFICAR" not in compiled:
                    compiled["SIN_CLASIFICAR"] = {k: [] for k in _DEF_KEYS}
                return compiled
            LOG.warning("El .py no define OCR_PATTERN/OCR_PATTERN_RAW. Intentando parser tolerante...")
        except Exception as e:
            LOG.warning("Fallo import módulo patrones (%s). Parser tolerante...", e)

        # parser tolerante de listas simples
        text = p.read_text(encoding="utf-8", errors="ignore")
        patterns: Dict[str, List[str]] = {}
        cur_label: Optional[str] = None
        in_list = False
        for line in text.splitlines():
            if not in_list:
                m = _LABEL_START.search(line)
                if m:
                    cur_label = m.group(2)
                    patterns.setdefault(cur_label, [])
                    in_list = True
            else:
                if _LIST_END.search(line):
                    cur_label = None
                    in_list = False
                    continue
                for m in _COMPILE_CALL.finditer(line):
                    patterns[cur_label].append(m.group(2))
        raw = {lbl: {"any": [re.compile(pat, _COMPILE_FLAGS) for pat in lst]} for lbl, lst in patterns.items()}
        if "SIN_CLASIFICAR" not in raw:
            raw["SIN_CLASIFICAR"] = {k: [] for k in _DEF_KEYS}
        LOG.info("Parser tolerante .py -> etiquetas cargadas: %d", len(raw))
        return raw

    elif ext == ".json":
        raw = json.loads(p.read_text(encoding="utf-8"))
        return _ensure_patterns_dict(raw)
    elif ext in (".yml", ".yaml"):
        if yaml is None:
            raise RuntimeError("Instala pyyaml para .yml/.yaml: pip install pyyaml")
        raw = yaml.safe_load(p.read_text(encoding="utf-8"))
        return _ensure_patterns_dict(raw)
    elif ext == ".csv":
        df = pd.read_csv(p)
        cols = {c.lower(): c for c in df.columns}
        if not {"label", "pattern"}.issubset(set(cols)):
            raise ValueError("CSV debe tener columnas: label, pattern")
        grouped = df.groupby(df[cols["label"]], dropna=False)[cols["pattern"]].apply(list).to_dict()
        raw = {lbl: {"any": [re.compile(s, _COMPILE_FLAGS) for s in lst]} for lbl, lst in grouped.items()}
        if "SIN_CLASIFICAR" not in raw:
            raw["SIN_CLASIFICAR"] = {k: [] for k in _DEF_KEYS}
        return raw
    else:
        raise ValueError(f"Extensión no soportada para patrones: {ext}")


In [20]:
#%%
RegexList = List[re.Pattern]
PatternsDict = Dict[str, Dict[str, RegexList]]  # {label: {any:[], must:[], exclude:[]}}

_COMPILE_FLAGS = re.IGNORECASE | re.UNICODE
_DEF_KEYS = ("any", "must", "exclude")

# Parser tolerante para .py con re.compile "roto" (re,compile / re.compil...)
_COMPILE_CALL = re.compile(r"re\W*c\w*?le\s*\(\s*(r?[\"'])(.+?)\1", re.IGNORECASE | re.UNICODE)
_LABEL_START  = re.compile(r"^\s*([\"'])([^\"']+)\1\s*:\s*\[", re.UNICODE)
_LIST_END     = re.compile(r"^\s*\],?\s*$")

def _ensure_patterns_dict(raw: Dict[str, Any]) -> PatternsDict:
    out: PatternsDict = {}
    for label, value in raw.items():
        d = {k: [] for k in _DEF_KEYS}
        if isinstance(value, dict):
            # {'any': [...], 'must': [...], 'exclude': [...]}
            for k in _DEF_KEYS:
                lst = value.get(k) or []
                d[k] = [re.compile(p, _COMPILE_FLAGS) if isinstance(p, str) else p for p in lst]
        elif isinstance(value, list):
            d["any"] = [re.compile(p, _COMPILE_FLAGS) if isinstance(p, str) else p for p in value]
        else:
            continue
        out[label] = d
    if "SIN_CLASIFICAR" not in out:
        out["SIN_CLASIFICAR"] = {k: [] for k in _DEF_KEYS}
    return out

def load_patterns(patterns_path: str | Path) -> PatternsDict:
    p = Path(patterns_path)
    if not p.exists():
        raise FileNotFoundError(f"No existe el archivo de patrones: {p}")
    ext = p.suffix.lower()

    if ext == ".py":
        # import limpio
        try:
            import importlib.util
            spec = importlib.util.spec_from_file_location("user_patterns", str(p))
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            if hasattr(mod, "OCR_PATTERN") and isinstance(mod.OCR_PATTERN, dict):
                LOG.info("Patrones PY: usando OCR_PATTERN del módulo.")
                return _ensure_patterns_dict(mod.OCR_PATTERN)
            if hasattr(mod, "OCR_PATTERN_RAW") and isinstance(mod.OCR_PATTERN_RAW, dict):
                LOG.info("Patrones PY: usando OCR_PATTERN_RAW.")
                compiled = {lbl: {k: [] for k in _DEF_KEYS} for lbl in mod.OCR_PATTERN_RAW}
                for lbl, lst in mod.OCR_PATTERN_RAW.items():
                    compiled[lbl]["any"] = [re.compile(s, _COMPILE_FLAGS) for s in lst]
                if "SIN_CLASIFICAR" not in compiled:
                    compiled["SIN_CLASIFICAR"] = {k: [] for k in _DEF_KEYS}
                return compiled
            LOG.warning("El .py no define OCR_PATTERN/OCR_PATTERN_RAW. Intentando parser tolerante...")
        except Exception as e:
            LOG.warning("Fallo import módulo patrones (%s). Parser tolerante...", e)

        # parser tolerante de listas simples
        text = p.read_text(encoding="utf-8", errors="ignore")
        patterns: Dict[str, List[str]] = {}
        cur_label: Optional[str] = None
        in_list = False
        for line in text.splitlines():
            if not in_list:
                m = _LABEL_START.search(line)
                if m:
                    cur_label = m.group(2)
                    patterns.setdefault(cur_label, [])
                    in_list = True
            else:
                if _LIST_END.search(line):
                    cur_label = None
                    in_list = False
                    continue
                for m in _COMPILE_CALL.finditer(line):
                    patterns[cur_label].append(m.group(2))
        raw = {lbl: {"any": [re.compile(pat, _COMPILE_FLAGS) for pat in lst]} for lbl, lst in patterns.items()}
        if "SIN_CLASIFICAR" not in raw:
            raw["SIN_CLASIFICAR"] = {k: [] for k in _DEF_KEYS}
        LOG.info("Parser tolerante .py -> etiquetas cargadas: %d", len(raw))
        return raw

    elif ext == ".json":
        raw = json.loads(p.read_text(encoding="utf-8"))
        return _ensure_patterns_dict(raw)
    elif ext in (".yml", ".yaml"):
        if yaml is None:
            raise RuntimeError("Instala pyyaml para .yml/.yaml: pip install pyyaml")
        raw = yaml.safe_load(p.read_text(encoding="utf-8"))
        return _ensure_patterns_dict(raw)
    elif ext == ".csv":
        df = pd.read_csv(p)
        cols = {c.lower(): c for c in df.columns}
        if not {"label", "pattern"}.issubset(set(cols)):
            raise ValueError("CSV debe tener columnas: label, pattern")
        grouped = df.groupby(df[cols["label"]], dropna=False)[cols["pattern"]].apply(list).to_dict()
        raw = {lbl: {"any": [re.compile(s, _COMPILE_FLAGS) for s in lst]} for lbl, lst in grouped.items()}
        if "SIN_CLASIFICAR" not in raw:
            raw["SIN_CLASIFICAR"] = {k: [] for k in _DEF_KEYS}
        return raw
    else:
        raise ValueError(f"Extensión no soportada para patrones: {ext}")


In [21]:
#%%
class OCREngine:
    def __init__(self, engine: str = "auto", lang: str = "spa", tesseract_cmd: Optional[str] = None):
        self.engine = engine
        self.lang = lang
        self._paddle = None

        if tesseract_cmd and pytesseract is not None:
            try:
                pytesseract.pytesseract.tesseract_cmd = tesseract_cmd
            except Exception:
                pass

        if engine in ("auto", "paddle") and PaddleOCR is not None:
            try:
                self._paddle = PaddleOCR(use_angle_cls=True, lang="es", use_gpu=True)  # usa GPU si hay
                LOG.info("PaddleOCR inicializado (GPU=True)")
            except Exception as e:
                LOG.warning("No se pudo inicializar PaddleOCR (%s). Se usará Tesseract si está disponible.", e)
                self._paddle = None
        elif engine == "paddle" and PaddleOCR is None:
            LOG.warning("PaddleOCR no instalado. Cambiando a Tesseract si disponible.")

    def ocr_tesseract(self, pil_img: Image.Image) -> Tuple[str, Optional[float]]:
        if pytesseract is None:
            return "", None
        cfg = "--psm 6"  # formularios/mixto
        try:
            raw = pytesseract.image_to_string(pil_img, lang=self.lang, config=cfg)
        except Exception as e:
            LOG.debug("Tesseract falló: %s", e)
            raw = ""
        return raw, None

    def ocr_paddle(self, pil_img: Image.Image) -> Tuple[str, Optional[float]]:
        if self._paddle is None or np is None or cv2 is None:
            return "", None
        try:
            arr = np.array(pil_img)
            if arr.ndim == 2:
                arr = cv2.cvtColor(arr, cv2.COLOR_GRAY2BGR)
            elif arr.shape[2] == 4:
                arr = cv2.cvtColor(arr, cv2.COLOR_RGBA2BGR)
            result = self._paddle.ocr(arr)
            lines, confs = [], []
            for block in result:
                for line in block:
                    lines.append(line[1][0])
                    confs.append(float(line[1][1]))
            text = " ".join(lines)
            conf_avg = float(np.mean(confs)) if confs else None
            return text, conf_avg
        except Exception as e:
            LOG.debug("PaddleOCR falló: %s", e)
            return "", None

    def ocr(self, pil_img: Image.Image) -> Tuple[str, str, Optional[float]]:
        if self.engine == "paddle":
            txt, c = self.ocr_paddle(pil_img)
            if txt.strip():
                return txt, "paddle", c
            txt, _ = self.ocr_tesseract(pil_img)
            return txt, "tesseract", None
        elif self.engine == "tesseract":
            txt, _ = self.ocr_tesseract(pil_img)
            return txt, "tesseract", None
        else:  # auto
            txt1, _ = self.ocr_tesseract(pil_img)
            if len(normalize_text(txt1)) >= 15:
                return txt1, "tesseract", None
            txt2, c2 = self.ocr_paddle(pil_img)
            if len(normalize_text(txt2)) > len(normalize_text(txt1)):
                return txt2, "paddle", c2
            return txt1, "tesseract", None


def _preprocess_for_ocr(pix) -> Image.Image:
    """fitz.Pixmap -> PIL.Image con limpieza (si OpenCV disponible)."""
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    if cv2 is None or np is None:
        return img
    try:
        arr = np.array(img)
        gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
        blur = cv2.GaussianBlur(gray, (3, 3), 0)
        th = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY, 31, 10)
        ker = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
        opened = cv2.morphologyEx(th, cv2.MORPH_OPEN, ker, iterations=1)
        return Image.fromarray(opened)
    except Exception:
        return img


def extract_page_text(page, *, ocr_engine: OCREngine, dpi: int = 250, min_len_for_text: int = 30) -> Tuple[str, str, Optional[float]]:
    """Texto embebido -> OCR. Retorna (texto_normalizado, fuente, conf)."""
    raw = page.get_text("text") or ""
    nrm = normalize_text(raw)
    if len(nrm) >= min_len_for_text:
        return nrm, "pdf_text", None

    mat = fitz.Matrix(dpi / 72.0, dpi / 72.0)
    pix = page.get_pixmap(matrix=mat, alpha=False)
    pil_img = _preprocess_for_ocr(pix)

    txt, source, conf = ocr_engine.ocr(pil_img)
    nrm2 = normalize_text(txt)
    return nrm2, source, conf


In [22]:
#%%
def build_pattern_weights(patterns: PatternsDict) -> Dict[str, float]:
    """Menor peso a patrones compartidos por varias etiquetas; mayor a los únicos."""
    inv: Dict[str, int] = {}
    for lbl, d in patterns.items():
        if lbl == "SIN_CLASIFICAR":
            continue
        for k in ("any", "must"):
            for rgx in d.get(k, []):  # <-- .get evita KeyError
                pat = getattr(rgx, "pattern", None)
                if not pat:
                    continue
                inv[pat] = inv.get(pat, 0) + 1

    weights: Dict[str, float] = {}
    for pat, cnt in inv.items():
        w = 1.0 if cnt == 1 else (0.6 if cnt == 2 else 0.4)
        weights[pat] = w
    return weights


class Classifier:
    def __init__(self, patterns: PatternsDict, min_score: float = 1.0, min_unique: int = 1, ambiguous_margin: float = 0.15):
        self.patterns = patterns
        self.min_score = float(min_score)
        self.min_unique = int(min_unique)
        self.ambiguous_margin = float(ambiguous_margin)
        self.weights = build_pattern_weights(patterns)

    def _score_label(self, text: str, d: Dict[str, RegexList]) -> Tuple[float, int, Optional[str]]:
        score = 0.0
        uniq = 0
        firstpat = None
        for rgx in d.get("must", []):
            if not rgx.search(text):
                return 0.0, 0, None
        for rgx in d.get("exclude", []):
            if rgx.search(text):
                return 0.0, 0, None
        for rgx in d.get("any", []):
            matched = False
            for _ in rgx.finditer(text):
                matched = True
                w = self.weights.get(rgx.pattern, 1.0)
                score += w
                if firstpat is None:
                    firstpat = rgx.pattern
            if matched and self.weights.get(rgx.pattern, 1.0) >= 0.9:
                uniq += 1
        return score, uniq, firstpat

    def classify(self, text: str) -> Tuple[str, float, Optional[str], bool]:
        best = ("SIN_CLASIFICAR", 0.0, None)
        second = ("", 0.0, None)
        for label, d in self.patterns.items():
            if label == "SIN_CLASIFICAR":
                continue
            score, uniq, firstpat = self._score_label(text, d)
            if score == 0.0 or uniq < self.min_unique:
                continue
            if score > best[1]:
                second = best
                best = (label, score, firstpat)
            elif score > second[1]:
                second = (label, score, firstpat)
        if best[1] < self.min_score:
            return "SIN_CLASIFICAR", 0.0, None, False
        ambiguous = second[1] > 0 and (best[1] - second[1]) <= self.ambiguous_margin
        return best[0], best[1], best[2], ambiguous


In [23]:
#%%
def _iter_pdfs(input_dir: str | Path, recursive: bool = True) -> Iterable[Path]:
    p = Path(input_dir)
    return p.rglob("*.pdf") if recursive else p.glob("*.pdf")

def _safe_open_pdf(pdf_path: Path):
    try:
        return fitz.open(pdf_path)
    except Exception as e:
        LOG.error("No se pudo abrir el PDF: %s (%s)", pdf_path, e)
        return None

def ensure_excel_path(p) -> Path:
    DEFAULT_OUTPUT_NAME = "inventario_ocr.xlsx"
    p = Path(p)
    if p.suffix.lower() == ".xlsx":
        p.parent.mkdir(parents=True, exist_ok=True)
        return p
    if not p.exists():
        p.mkdir(parents=True, exist_ok=True)
        return p / DEFAULT_OUTPUT_NAME
    if p.is_dir():
        return p / DEFAULT_OUTPUT_NAME
    return p.with_suffix(".xlsx")

def classify_pdf(
    pdf_path: str | Path,
    clf: Classifier,
    ocr_engine: OCREngine,
    dpi: int = 250,
    min_len_for_text: int = 30,
) -> pd.DataFrame:
    pdf_path = Path(pdf_path)
    doc = _safe_open_pdf(pdf_path)
    if doc is None:
        return pd.DataFrame()

    total = doc.page_count
    rows = []
    iterator = range(total)
    if tqdm is not None:
        iterator = tqdm(iterator, desc=f"{pdf_path.name}", leave=False)

    for i in iterator:
        try:
            page = doc.load_page(i)
            text, source, conf = extract_page_text(page, ocr_engine=ocr_engine, dpi=dpi, min_len_for_text=min_len_for_text)
            label, score, firstpat, ambiguous = clf.classify(text)
            rows.append({
                "pdf_name": pdf_path.name,
                "pdf_path": str(pdf_path),
                "total_pages": total,
                "page": i + 1,
                "source": source,
                "ocr_conf": conf if conf is not None else "",
                "label": label if not ambiguous else f"{label} (AMB)",
                "score": round(float(score), 3),
                "first_match": firstpat or "",
                "text_preview": text[:300],
            })
        except Exception as e:
            LOG.warning("Fallo en %s página %d: %s", pdf_path.name, i+1, e)
            rows.append({
                "pdf_name": pdf_path.name,
                "pdf_path": str(pdf_path),
                "total_pages": total,
                "page": i + 1,
                "source": "error",
                "ocr_conf": "",
                "label": "SIN_CLASIFICAR",
                "score": 0,
                "first_match": "",
                "text_preview": "",
            })

    doc.close()
    return pd.DataFrame(rows)

def run_folder(
    input_dir: str | Path,
    output_excel: str | Path,
    patterns: PatternsDict,
    recursive: bool = True,
    engine: str = "auto",
    lang: str = "spa",
    dpi: int = 250,
    min_len_for_text: int = 30,
    min_score_to_accept: float = 1.0,
    min_unique: int = 1,
    ambiguous_margin: float = 0.15,
    tesseract_cmd: Optional[str] = None,
) -> pd.DataFrame:
    input_dir = Path(input_dir)
    output_excel = ensure_excel_path(output_excel)
    output_excel.parent.mkdir(parents=True, exist_ok=True)

    ocr_engine = OCREngine(engine=engine, lang=lang, tesseract_cmd=tesseract_cmd)
    clf = Classifier(patterns, min_score=min_score_to_accept, min_unique=min_unique, ambiguous_margin=ambiguous_margin)

    all_rows = []
    pdfs = list(_iter_pdfs(input_dir, recursive=recursive))
    LOG.info("Encontrados %d PDF(s) en '%s'", len(pdfs), input_dir)

    iterator = pdfs
    if tqdm is not None:
        iterator = tqdm(pdfs, desc="Clasificando PDFs")

    for pdf_path in iterator:
        df_one = classify_pdf(
            pdf_path=pdf_path,
            clf=clf,
            ocr_engine=ocr_engine,
            dpi=dpi,
            min_len_for_text=min_len_for_text,
        )
        if not df_one.empty:
            all_rows.append(df_one)

    if not all_rows:
        LOG.warning("No se generaron filas. ¿Hay PDFs válidos en la carpeta?")
        result = pd.DataFrame(columns=[
            "pdf_name","pdf_path","total_pages","page","source","ocr_conf","label","score","first_match","text_preview"
        ])
    else:
        result = pd.concat(all_rows, ignore_index=True)

    # Guardar Excel
    try:
        with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:
            result.to_excel(writer, index=False, sheet_name="inventario")
            resumen = result.groupby("label", dropna=False).size().reset_index(name="paginas")
            resumen.to_excel(writer, index=False, sheet_name="resumen_etiquetas")
    except PermissionError as e:
        raise PermissionError(
            f"No pude escribir '{output_excel}'. Cierra el archivo si está abierto y confirma permisos."
        ) from e

    LOG.info("Listo. Excel guardado en: %s", output_excel)
    return result


In [24]:
#%%
# Cargar patrones y correr
patterns = load_patterns(PATTERNS_FILE)

df_result = run_folder(
    input_dir=INPUT_DIR,
    output_excel=OUTPUT_XLSX,
    patterns=patterns,
    recursive=RECURSIVE,
    engine=ENGINE,
    lang=LANG,
    dpi=DPI,
    min_len_for_text=MIN_LEN_FOR_TEXT,
    min_score_to_accept=MIN_SCORE,
    min_unique=MIN_UNIQUE,
    ambiguous_margin=AMBIG_MARGIN,
    tesseract_cmd=TESSERACT_CMD,
)

# Vista rápida
df_result.head(20)


[WARNING] Fallo import módulo patrones (compile() missing required argument 'mode' (pos 3)). Parser tolerante...
[INFO] Parser tolerante .py -> etiquetas cargadas: 50
[INFO] Encontrados 12 PDF(s) en 'D:\historias\_01_historias\LETRA A\ACTA N° 70\1'


Clasificando PDFs: 100%|██████████| 12/12 [00:06<00:00,  1.97it/s]


[INFO] Listo. Excel guardado en: C:\Users\juans\Documents\version_final_historias laborales\answer_ocr\inventario_ocr.xlsx


,pdf_name,pdf_path,total_pages,page,source,ocr_conf,label,score,first_match,text_preview
0,Abad Abad Rosa Elena.pdf,D:\historias\_01_historias\LETRA A\ACTA N° 70\...,8,1,pdf_text,,SIN_CLASIFICAR,0.0,,^e^a/zica i/e 8 e/e oficio no. 794 jtilio n a#...
1,Abad Abad Rosa Elena.pdf,D:\historias\_01_historias\LETRA A\ACTA N° 70\...,8,2,pdf_text,,SIN_CLASIFICAR,0.0,,", s^^e^a'ea t/e ^ee/om^ia w f/e s^e^/feae/f./a..."
2,Abad Abad Rosa Elena.pdf,D:\historias\_01_historias\LETRA A\ACTA N° 70\...,8,3,pdf_text,,SIN_CLASIFICAR,0.0,,^ ( / ^ nal a^¿?s/9 n/5/9ri /^/3/9?) cama nomb...
3,Abad Abad Rosa Elena.pdf,D:\historias\_01_historias\LETRA A\ACTA N° 70\...,8,4,pdf_text,,SIN_CLASIFICAR,0.0,,registro del empleado dbcreto o resolucion sue...
4,Abad Abad Rosa Elena.pdf,D:\historias\_01_historias\LETRA A\ACTA N° 70\...,8,5,pdf_text,,SIN_CLASIFICAR,0.0,,forma publicaciones c.n.p. caja nacional de pr...
5,Abad Abad Rosa Elena.pdf,D:\historias\_01_historias\LETRA A\ACTA N° 70\...,8,6,pdf_text,,SIN_CLASIFICAR,0.0,,5! caja nacional de prevision servicio medico ...
6,Abad Abad Rosa Elena.pdf,D:\historias\_01_historias\LETRA A\ACTA N° 70\...,8,7,pdf_text,,SIN_CLASIFICAR,0.0,,camara de representantes acta de posesion 351 ...
7,Abad Abad Rosa Elena.pdf,D:\historias\_01_historias\LETRA A\ACTA N° 70\...,8,8,pdf_text,,SIN_CLASIFICAR,0.0,,• ^ r w/mm-mm im||ip» ::rj (c(bmwm¥m(^ mb tpws...
8,Abad Buelvas Nelson Elias.pdf,D:\historias\_01_historias\LETRA A\ACTA N° 70\...,23,1,pdf_text,,SIN_CLASIFICAR,0.0,,-“7*1 1 - a^' . t {. i- -r-*ir ^¿¿'/na^riz' e/...
9,Abad Buelvas Nelson Elias.pdf,D:\historias\_01_historias\LETRA A\ACTA N° 70\...,23,2,pdf_text,,SIN_CLASIFICAR,0.0,,"ad,;„wi*pp^-.... i-.70 s /;m 2- •-. ■ -.-a? >:..."
